# GitHub Models: news-event extraction comparison

A small, separate model-quality check. It reads GITHUB_API from .env or env.txt, tests a few account-available models on four deliberately different articles, and does not change the Gemini notebook or write results files.

## 1. Setup

If required, run: %pip install -U openai python-dotenv pandas

In [4]:
from __future__ import annotations

import json
import os
from pathlib import Path
from time import perf_counter

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

search_roots = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'Sentiment project']
for root in dict.fromkeys(search_roots):
    for candidate in (root / '.env', root / 'env.txt'):
        if candidate.exists():
            load_dotenv(candidate)

GITHUB_API = os.getenv('GITHUB_API')
assert GITHUB_API, 'GITHUB_API is missing. Add it to .env or env.txt; do not paste it into the notebook.'

client = OpenAI(base_url='https://models.github.ai/inference', api_key=GITHUB_API, timeout=12.0, max_retries=0)
print('GitHub Models client configured. API key loaded:', bool(GITHUB_API))

GitHub Models client configured. API key loaded: True


## 2. Discover and select models

The catalogue is account-dependent. The notebook first tries to list it, then records unavailable models as errors rather than stopping.

In [5]:
MODEL_CANDIDATES = [
    'openai/gpt-4o-mini',
    'meta/llama-3.3-70b-instruct',
]

try:
    available_models = sorted(model.id for model in client.models.list().data)
    print(f'Discovered {len(available_models)} model IDs.')
    display(pd.DataFrame({'available_model': available_models}))
except Exception as exc:
    available_models = None
    print(f'Model discovery unavailable ({type(exc).__name__}): {exc}')
    print('The comparison will try candidate IDs individually.')

models_to_test = ([model for model in MODEL_CANDIDATES if model in available_models]
                  if available_models is not None else MODEL_CANDIDATES)
print('Models selected for test:', models_to_test)

Model discovery unavailable (NotFoundError): 404 page not found
The comparison will try candidate IDs individually.
Models selected for test: ['openai/gpt-4o-mini', 'meta/llama-3.3-70b-instruct']


## 3. Mixed article audit set

The cases include direct company news, indirect mentions, and ETF/background coverage. Replace them with sampled article-table rows after the API route is proven.

In [6]:
TEST_ARTICLES = [
    {'article_uid': 'test_tsmc_explainer', 'ticker': 'AAPL', 'title': 'What is TSMC?', 'source': 'Tech Monitor',
     'summary': 'An explainer describes Taiwan Semiconductor Manufacturing Company and its role as a chip manufacturer for major technology firms, including Apple.', 'expected_relevant': False},
    {'article_uid': 'test_aapl_market_value', 'ticker': 'AAPL', 'title': 'Apple’s market value drops below $2 trillion', 'source': 'Al Jazeera',
     'summary': 'Apple’s market capitalisation fell below $2 trillion amid concerns about inflation and a slowing economy.', 'expected_relevant': True},
    {'article_uid': 'test_etf_volume', 'ticker': 'AAPL', 'title': "Tuesday's ETF with Unusual Volume: IWL", 'source': 'Nasdaq',
     'summary': 'A market note discusses unusual trading volume in an ETF that holds Apple among many other constituents.', 'expected_relevant': False},
    {'article_uid': 'test_luxshare_supply_chain', 'ticker': 'AAPL', 'title': 'Apple to sign Luxshare for iPhone production in China - FT', 'source': 'Reuters',
     'summary': 'Apple is reported to be preparing Luxshare Precision to assemble premium iPhone models in China as it diversifies suppliers.', 'expected_relevant': True},
]
display(pd.DataFrame(TEST_ARTICLES)[['article_uid', 'ticker', 'title', 'source', 'expected_relevant']])

,article_uid,ticker,title,source,expected_relevant
0,test_tsmc_explainer,AAPL,What is TSMC?,Tech Monitor,False
1,test_aapl_market_value,AAPL,Apple’s market value drops below $2 trillion,Al Jazeera,True
2,test_etf_volume,AAPL,Tuesday's ETF with Unusual Volume: IWL,Nasdaq,False
3,test_luxshare_supply_chain,AAPL,Apple to sign Luxshare for iPhone production i...,Reuters,True


## 4. Fixed extraction contract

Every model receives exactly the same prompt. Rationale is a short audit explanation, not hidden chain-of-thought. No price data or future outcomes are supplied.

In [7]:
SYSTEM_PROMPT = '''You are a careful financial-news event extractor. Analyse one supplied article for one target ticker using only the supplied text.

Mark relevant=true only if the article states a concrete causal mechanism affecting the target company's operations, revenue, costs, strategy, legal position, supply chain, or investor-relevant company event. A direct ticker-specific share-price or market-capitalisation event is relevant even when its driver is macroeconomic. A ticker mention in an ETF, listicle, comparison, generic market commentary, or background explainer is not enough.

Do not use future information. Do not predict price returns. Return JSON only with exactly these fields:
- relevant: boolean
- event_type: one of [market_sentiment, earnings, guidance, product, supply_chain, regulation, legal, management, macro, other, none]
- direction: one of [positive, negative, neutral, risk_on, risk_off, none]
- semantic_materiality: one of [none, low, medium, high, very_high]
- novelty: one of [none, low, medium, high, very_high]
- confidence: one of [low, medium, high, very_high]
- rationale: a single concise sentence naming the concrete ticker-specific mechanism, or explaining why there is none.

For irrelevant articles use event_type=none, direction=none, semantic_materiality=none, novelty=none, confidence=low.'''

def article_prompt(article: dict) -> str:
    return json.dumps({key: article[key] for key in ('article_uid', 'ticker', 'title', 'source', 'summary')}, ensure_ascii=False, indent=2)

## 5. Run comparison

At most five models multiplied by four articles are called. Start with two models if you want the smallest free-tier test. Any unavailable model stays in the result table as an error row.

In [8]:
MAX_MODELS = 2
MAX_ARTICLES = 4
RUN_COMPARISON = True

def parse_json_object(text: str) -> dict:
    text = text.strip()
    fence = chr(96) * 3
    if text.startswith(fence):
        text = text.split('\n', 1)[-1].rsplit(fence, 1)[0].strip()
    start, end = text.find('{'), text.rfind('}')
    if start == -1 or end == -1:
        raise ValueError('No JSON object found in model response.')
    return json.loads(text[start:end + 1])

results = []
if RUN_COMPARISON:
    for model in models_to_test[:MAX_MODELS]:
        for article in TEST_ARTICLES[:MAX_ARTICLES]:
            started = perf_counter()
            base_row = {'model': model, 'article_uid': article['article_uid'], 'title': article['title'], 'expected_relevant': article['expected_relevant']}
            try:
                response = client.chat.completions.create(
                    model=model,
                    temperature=0,
                    messages=[{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': article_prompt(article)}],
                )
                raw_text = response.choices[0].message.content or ''
                parsed = parse_json_object(raw_text)
                results.append({**base_row, **parsed, 'status': 'ok', 'latency_seconds': round(perf_counter() - started, 2), 'raw_response': raw_text})
            except Exception as exc:
                results.append({**base_row, 'status': 'error', 'error_type': type(exc).__name__, 'error_message': str(exc), 'latency_seconds': round(perf_counter() - started, 2)})

results_df = pd.DataFrame(results)
display(results_df)

,model,article_uid,title,expected_relevant,relevant,event_type,direction,semantic_materiality,novelty,confidence,rationale,status,latency_seconds,raw_response
0,openai/gpt-4o-mini,test_tsmc_explainer,What is TSMC?,False,False,none,none,none,none,low,The article is an explainer about TSMC and doe...,ok,2.01,"{\n ""relevant"": false,\n ""event_type"": ""none..."
1,openai/gpt-4o-mini,test_aapl_market_value,Apple’s market value drops below $2 trillion,True,True,macro,negative,medium,low,medium,The article states that Apple's market capital...,ok,1.98,"{\n ""relevant"": true,\n ""event_type"": ""macro..."
2,openai/gpt-4o-mini,test_etf_volume,Tuesday's ETF with Unusual Volume: IWL,False,False,none,none,none,none,low,The article discusses ETF volume but does not ...,ok,2.05,"{\n ""relevant"": false,\n ""event_type"": ""none..."
3,openai/gpt-4o-mini,test_luxshare_supply_chain,Apple to sign Luxshare for iPhone production i...,True,True,supply_chain,neutral,medium,medium,medium,Apple is diversifying its suppliers by prepari...,ok,1.98,"{\n ""relevant"": true,\n ""event_type"": ""suppl..."
4,meta/llama-3.3-70b-instruct,test_tsmc_explainer,What is TSMC?,False,False,none,none,none,none,low,The article is a generic explainer about TSMC ...,ok,1.77,"{\n ""relevant"": false,\n ""event_type"": ""none..."
5,meta/llama-3.3-70b-instruct,test_aapl_market_value,Apple’s market value drops below $2 trillion,True,True,market_sentiment,negative,high,low,high,Apple's market capitalisation fell below $2 tr...,ok,1.57,"```json\n{\n ""relevant"": true,\n ""event_type..."
6,meta/llama-3.3-70b-instruct,test_etf_volume,Tuesday's ETF with Unusual Volume: IWL,False,False,none,none,none,none,low,The article only mentions AAPL as a constituen...,ok,1.63,"```\n{\n ""relevant"": false,\n ""event_type"": ..."
7,meta/llama-3.3-70b-instruct,test_luxshare_supply_chain,Apple to sign Luxshare for iPhone production i...,True,True,supply_chain,positive,medium,medium,high,Apple is diversifying its suppliers by prepari...,ok,1.08,"```json\n{\n ""relevant"": true,\n ""event_type..."


## 6. Inspect agreement and failures

This is a compact audit, not a statistical benchmark. Read the row-level rationales before choosing a production model. A suitable model should reject the TSMC and ETF examples while identifying the Apple supply-chain event.

In [9]:
if results_df.empty:
    print('No comparison results yet. Set RUN_COMPARISON=True and run the preceding cell.')
else:
    successful = results_df.loc[results_df['status'].eq('ok')].copy()
    if not successful.empty:
        successful['relevance_correct'] = successful['relevant'].eq(successful['expected_relevant'])
        summary = (successful.groupby('model', as_index=False)
                   .agg(successful_calls=('status', 'size'), relevance_accuracy=('relevance_correct', 'mean'), mean_latency_seconds=('latency_seconds', 'mean'))
                   .sort_values(['relevance_accuracy', 'mean_latency_seconds'], ascending=[False, True]))
        display(summary)
        display(successful[['model', 'title', 'relevant', 'expected_relevant', 'event_type', 'direction', 'semantic_materiality', 'novelty', 'confidence', 'rationale']])
    failures = results_df.loc[results_df['status'].eq('error')]
    if not failures.empty:
        print('Unavailable or failed calls:')
        display(failures[['model', 'title', 'error_type', 'error_message']])

,model,successful_calls,relevance_accuracy,mean_latency_seconds
0,meta/llama-3.3-70b-instruct,4,1.0,1.5125
1,openai/gpt-4o-mini,4,1.0,2.0050


,model,title,relevant,expected_relevant,event_type,direction,semantic_materiality,novelty,confidence,rationale
0,openai/gpt-4o-mini,What is TSMC?,False,False,none,none,none,none,low,The article is an explainer about TSMC and doe...
1,openai/gpt-4o-mini,Apple’s market value drops below $2 trillion,True,True,macro,negative,medium,low,medium,The article states that Apple's market capital...
2,openai/gpt-4o-mini,Tuesday's ETF with Unusual Volume: IWL,False,False,none,none,none,none,low,The article discusses ETF volume but does not ...
3,openai/gpt-4o-mini,Apple to sign Luxshare for iPhone production i...,True,True,supply_chain,neutral,medium,medium,medium,Apple is diversifying its suppliers by prepari...
4,meta/llama-3.3-70b-instruct,What is TSMC?,False,False,none,none,none,none,low,The article is a generic explainer about TSMC ...
5,meta/llama-3.3-70b-instruct,Apple’s market value drops below $2 trillion,True,True,market_sentiment,negative,high,low,high,Apple's market capitalisation fell below $2 tr...
6,meta/llama-3.3-70b-instruct,Tuesday's ETF with Unusual Volume: IWL,False,False,none,none,none,none,low,The article only mentions AAPL as a constituen...
7,meta/llama-3.3-70b-instruct,Apple to sign Luxshare for iPhone production i...,True,True,supply_chain,positive,medium,medium,high,Apple is diversifying its suppliers by prepari...
